In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install trl
!pip install torchinfo
!pip install --upgrade "torchao>=0.16.0"
!pip install causal-conv1d>=1.2.0
!pip install mamba-ssm
import torch
import torch.nn as nn

from trl import SFTConfig, SFTTrainer

from torchinfo import summary
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_id = "state-spaces/mamba-130m-hf"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model1 = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map=device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  error: subprocess-exited-with-error
  
  × Building wheel for causal-conv1d (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from 

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.79k [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/517M [00:00<?, ?B/s]

[transformers] The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.


Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [3]:
model1.backbone.layers[0]

MambaBlock(
  (norm): MambaRMSNorm(768, eps=1e-05)
  (mixer): MambaMixer(
    (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
    (act): SiLUActivation()
    (in_proj): Linear(in_features=768, out_features=3072, bias=False)
    (x_proj): Linear(in_features=1536, out_features=80, bias=False)
    (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
    (out_proj): Linear(in_features=1536, out_features=768, bias=False)
  )
)

In [4]:
summary(model1, input_size=(1, 16), dtypes=[torch.long])

Layer (type:depth-idx)                             Output Shape              Param #
MambaForCausalLM                                   --                        --
├─MambaModel: 1-1                                  --                        --
│    └─Embedding: 2-1                              [1, 16, 768]              38,615,040
│    └─ModuleList: 2-2                             --                        --
│    │    └─MambaBlock: 3-1                        [1, 16, 768]              3,771,648
│    │    └─MambaBlock: 3-2                        [1, 16, 768]              3,771,648
│    │    └─MambaBlock: 3-3                        [1, 16, 768]              3,771,648
│    │    └─MambaBlock: 3-4                        [1, 16, 768]              3,771,648
│    │    └─MambaBlock: 3-5                        [1, 16, 768]              3,771,648
│    │    └─MambaBlock: 3-6                        [1, 16, 768]              3,771,648
│    │    └─MambaBlock: 3-7                        [1, 16, 768]  

In [5]:
prompt = "Explain the theory of relativity"


inputs = tokenizer(prompt, return_tensors="pt").to(device)

output_tokens = model1.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_k=50
)


result = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print(result)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPTNeoXTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Explain the theory of relativity.

Brain states: The human brain

The brain states: The brain states are the states of being in a certain mental state,


In [6]:
class SVDLinear(nn.Module):
    def __init__(self, in_features, out_features, rank, bias=True):

        super().__init__()
        self.v_layer = nn.Linear(in_features, rank, bias=False)
        self.u_layer = nn.Linear(rank, out_features, bias=bias)

    def forward(self, x):
        return self.u_layer(self.v_layer(x))
def apply_svd_to_linear(module, rank):
  for name, child in module.named_children():
    if isinstance(child, nn.Linear) and name in ["in_proj", "out_proj"]:

        in_features = child.in_features
        out_features = child.out_features


        current_rank = min(rank, in_features, out_features)


        W = child.weight.data.float()
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)

        U_r = U[:, :current_rank]
        S_r = S[:current_rank]
        Vh_r = Vh[:current_rank, :]


        svd_module = SVDLinear(in_features, out_features, current_rank, bias=(child.bias is not None))


        svd_module.v_layer.weight.data = (torch.diag(S_r) @ Vh_r).to(child.weight.dtype)
        svd_module.u_layer.weight.data = U_r.to(child.weight.dtype)

        if child.bias is not None:
            svd_module.u_layer.bias.data = child.bias.data.clone()


        setattr(module, name, svd_module)
    else:

        apply_svd_to_linear(child, rank)

apply_svd_to_linear(model1, rank=128)

In [7]:
summary(model1, input_size=(1, 16), dtypes=[torch.long])

Layer (type:depth-idx)                             Output Shape              Param #
MambaForCausalLM                                   --                        --
├─MambaModel: 1-1                                  --                        --
│    └─Embedding: 2-1                              [1, 16, 768]              38,615,040
│    └─ModuleList: 2-2                             --                        --
│    │    └─MambaBlock: 3-1                        [1, 16, 768]              1,019,136
│    │    └─MambaBlock: 3-2                        [1, 16, 768]              1,019,136
│    │    └─MambaBlock: 3-3                        [1, 16, 768]              1,019,136
│    │    └─MambaBlock: 3-4                        [1, 16, 768]              1,019,136
│    │    └─MambaBlock: 3-5                        [1, 16, 768]              1,019,136
│    │    └─MambaBlock: 3-6                        [1, 16, 768]              1,019,136
│    │    └─MambaBlock: 3-7                        [1, 16, 768]  

In [8]:
prompt = "Explain the theory of relativity"


inputs = tokenizer(prompt, return_tensors="pt").to(device)

output_tokens = model1.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_k=50
)


result = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print(result)

Explain the theory of relativityjà��������ometimes doxor^](# Palestinamssamss}{~}{~}{~- prior


In [9]:

print("Loading Wikitext Dataset...")

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-v1", split="train")


dataset = dataset.filter(lambda x: len(x['text'].strip()) > 20)

print(f"Dataset ready!:\n{dataset[10]['text']}")



Loading Wikitext Dataset...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-v1/test-00000-of-00001.parque(…):   0%|          | 0.00/685k [00:00<?, ?B/s]

wikitext-2-v1/train-00000-of-00001.parqu(…):   0%|          | 0.00/6.07M [00:00<?, ?B/s]

wikitext-2-v1/validation-00000-of-00001.(…):   0%|          | 0.00/618k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Dataset ready!:
 Concept work for Valkyria Chronicles III began after development finished on Valkyria Chronicles II in early 2010 , with full development beginning shortly after this . The director of Valkyria Chronicles II , Takeshi Ozawa , returned to that role for Valkyria Chronicles III . Development work took approximately one year . After the release of Valkyria Chronicles II , the staff took a look at both the popular response for the game and what they wanted to do next for the series . Like its predecessor , Valkyria Chronicles III was developed for PlayStation Portable : this was due to the team wanting to refine the mechanics created for Valkyria Chronicles II , and they had not come up with the " revolutionary " idea that would warrant a new entry for the PlayStation 3 . Speaking in an interview , it was stated that the development team considered Valkyria Chronicles III to be the series ' first true sequel : while Valkyria Chronicles II had required a large amount of tria

In [ ]:
###### POS tagging transformation of the dataset

!python -m spacy download en_core_web_sm
import spacy


nlp = spacy.load("en_core_web_sm")


def apply_pos_tags(example):

    doc = nlp(example['text'])
    tagged_words = []

    for token in doc:

        if token.text.strip():
            tagged_words.append(f"[{token.pos_}] {token.text}")

    return {"text": " ".join(tagged_words)}




tagged_dataset = dataset.map(apply_pos_tags, num_proc=4)

print("\nTagged dataset:")
print("-" * 50)
print(tagged_dataset[10]['text'][:500])
print("-" * 50)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 49.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Map (num_proc=4):   0%|          | 0/20741 [00:00<?, ? examples/s]

In [ ]:
split_dataset =tagged_dataset.train_test_split(test_size=0.05, seed=42)

### No tagging
#split_dataset =dataset.train_test_split(test_size=0.05, seed=42)
####

train_data = split_dataset["train"]
val_data = split_dataset["test"]

print(f"Train size: {len(train_data)} | Validation size: {len(val_data)}")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["u_layer", "v_layer"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    use_rslora=True,
    init_lora_weights="gaussian",
    modules_to_save=["lm_head"],
)

peft_model = get_peft_model(model1, lora_config)

peft_model.print_trainable_parameters()

In [ ]:
import os


current_rank = 128

sft_config = SFTConfig(
    output_dir=f"/content/drive/MyDrive/mamba-checkpoints-rank-{current_rank}-POS",
    run_name=f"mamba-wiki-text-rank-{current_rank}-POS",
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    max_steps=100,
    fp16=True,
    optim="adamw_torch",
    dataset_text_field="text",
    max_length=256,

)


trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=sft_config,
)

for param in peft_model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

trainer.train()

In [ ]:


save_path = f"/content/drive/MyDrive/Project_Generative_Weights/mamba-wiki-text-final-rank-{current_rank}-POS"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved on: {save_path}")

In [ ]:
import matplotlib.pyplot as plt


history = trainer.state.log_history

train_steps = []
train_losses = []
eval_steps = []
eval_losses = []


for entry in history:

    if "loss" in entry and "step" in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])

    elif "eval_loss" in entry and "step" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])


plt.figure(figsize=(10, 6))

if train_losses:
    plt.plot(train_steps, train_losses, label="Training Loss", color="blue", marker="o")


if eval_losses:
    plt.plot(eval_steps, eval_losses, label="Validation Loss", color="red", marker="x", linestyle="--")


plt.xlabel("Steps", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("Training vs Validation Loss", fontsize=14, fontweight="bold")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.7)


plt.show()

In [ ]:
#### No tagging prompt
peft_model.eval()


art_prompt = "Leonardo da Vinci was one of the greatest painters of the Renaissance. He is most famous for"


inputs = tokenizer(art_prompt, return_tensors="pt").to(device)


with torch.no_grad():
    outputs = peft_model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True,
        top_k=50,
        eos_token_id=tokenizer.eos_token_id
    )


result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

In [ ]:
import math

print("Evaluation...")


eval_results = trainer.evaluate()


eval_loss = eval_results["eval_loss"]


try:
    perplexity = math.exp(eval_loss)
except OverflowError:

    perplexity = float("inf")

print("\n" + "="*40)
print(f"Validation Loss: {eval_loss:.4f}")
print(f"Perplexity (PPL): {perplexity:.4f}")
print("="*40)

In [ ]:
import pandas as pd
import os
import math



run_name = sft_config.run_name

trainable_params, all_params = peft_model.get_nb_trainable_parameters()
original_params = 167750400
compression_percent = round((1 - (all_params / original_params)) * 100, 2)


history = trainer.state.log_history
eval_losses = [entry["eval_loss"] for entry in history if "eval_loss" in entry]
final_eval_loss = eval_losses[-1] if eval_losses else None


if final_eval_loss is not None:
    perplexity = math.exp(final_eval_loss)
else:
    perplexity = "N/A"


new_data = pd.DataFrame([{
    "Run ID": run_name,
    "SVD Rank": current_rank,
    "Total Params": all_params,
    "Trainable (LoRA)": trainable_params,
    "Compression %": f"{compression_percent}%",
    "Final Val Loss": round(final_eval_loss, 4) if final_eval_loss else "N/A",
    "Perplexity": round(perplexity, 2) if isinstance(perplexity, float) else perplexity
}])


csv_path = "/content/drive/MyDrive/mamba_svd_master_log.csv"

if os.path.exists(csv_path):

    new_data.to_csv(csv_path, mode='a', header=False, index=False)
    print(f"Rank {current_rank} | Perplexity: {new_data['Perplexity'][0]}")
else:

    new_data.to_csv(csv_path, index=False)
    print(f"New log: Rank {current_rank} | Perplexity: {new_data['Perplexity'][0]}")

In [ ]:
### Tagged prompt

import spacy
import torch


nlp = spacy.load("en_core_web_sm")
prompt_text = "Leonardo da Vinci was one of the greatest painters of the Renaissance. He is most famous for"


doc = nlp(prompt_text)
tagged_prompt = " ".join([f"[{token.pos_}] {token.text}" for token in doc if token.text.strip()])


print(tagged_prompt)
print("-" * 50)


peft_model.eval()


inputs = tokenizer(tagged_prompt, return_tensors="pt").to(device)


with torch.no_grad():
    output_tokens = peft_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id
    )


result = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print("\nFinal result:")
print(result)

In [ ]:
#from google.colab import runtime
#print("Η εκπαίδευση τελείωσε! Τερματισμός του session για εξοικονόμηση credits...")
#runtime.unassign()